# Momentum

Momentum strategies rank assets by recent price performance and buy recent winners.

Abbreviations used in this notebook:

- **MOM**: Momentum.
- **MA**: Moving Average.
- **SMA**: Simple Moving Average.
- **bps**: Basis points, where 100 bps equals 1 percent.
- **K**: Number of selected stocks in the portfolio.
- **IR**: Information Ratio.

## 1. Intuition

Momentum is based on the observation that price trends can persist. It is often explained by slow information diffusion, investor underreaction, and behavioral herding.

The risk is reversal: yesterday's winners can quickly become tomorrow's losers.

## 2. Mathematics

Lookback momentum:

$$
MOM_{i,t} = \frac{P_{i,t}}{P_{i,t-L}} - 1
$$

Top-rank portfolio return:

$$
R_{p,t} = \frac{1}{K}\sum_{i \in Winners} R_{i,t}
$$

Net return after transaction costs:

$$
R_{net,t} = R_{gross,t} - Turnover_t \times CostRate
$$

Where:
- $MOM_i,t$ = momentum signal for asset $i$ at time $t$.
- $P_i,t$ = price of asset $i$ at time $t$.
- $L$ = lookback length.
- $K$ = number of winners selected.
- $R_p,t$ = portfolio return at time $t$.
- $Turnover_t$ = portfolio weight change at time $t$.
- $\text{CostRate}$ = transaction cost rate.


## 3. Implementation

We form a monthly rebalanced portfolio from the top six stocks by 63-day momentum.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "05_strategies" / "strategy_utils.py"
spec = importlib.util.spec_from_file_location("strategy_utils", helper_path)
strategy_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy_utils)

plt.style.use("seaborn-v0_8-whitegrid")
prices, returns, fundamentals = strategy_utils.generate_strategy_universe()
benchmark = returns.mean(axis=1)

lookback = 63
top_n = 6
cost_rate = 0.0010
momentum = prices.pct_change(lookback)
rebalance_dates = returns.resample("ME").last().index
weights = pd.DataFrame(0.0, index=returns.index, columns=returns.columns)
current = pd.Series(1 / len(returns.columns), index=returns.columns)

for date in returns.index:
    if date in rebalance_dates and not momentum.loc[date].isna().all():
        winners = momentum.loc[date].sort_values(ascending=False).head(top_n).index
        current = pd.Series(0.0, index=returns.columns)
        current.loc[winners] = 1 / top_n
    weights.loc[date] = current

gross = (weights.shift(1).fillna(1 / len(returns.columns)) * returns).sum(axis=1)
turnover = weights.diff().abs().sum(axis=1).fillna(0)
net = gross - turnover * cost_rate

pd.DataFrame({"gross": gross, "net": net, "turnover": turnover}).head()

In [ ]:
summary = pd.DataFrame({
    "momentum_net": strategy_utils.performance_summary(net, benchmark),
    "benchmark": strategy_utils.performance_summary(benchmark),
})
summary.round(4)

## 4. Visualization

Momentum analysis should show both wealth and how the portfolio rotates through winners.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
(1 + pd.DataFrame({"Momentum": net, "Benchmark": benchmark})).cumprod().plot(ax=axes[0], color=["#2f6f8f", "#9a6b2f"])
axes[0].set_title("Momentum Strategy vs Benchmark")
axes[0].set_ylabel("Growth of 1")

turnover.plot(ax=axes[1], color="#2f6f8f")
axes[1].set_title("Portfolio Turnover")
axes[1].set_ylabel("Turnover")
axes[1].set_xlabel("Date")
plt.tight_layout(); plt.show()

## 5. Application

Momentum strategies require careful cost control because frequent rebalancing can erode returns. They also need risk controls because crowded momentum trades can unwind sharply.

In [ ]:
latest_momentum = momentum.iloc[-1].sort_values(ascending=False).head(10).to_frame("63_day_momentum")
latest_momentum

## 6. Reflection

- Momentum buys strength rather than cheapness.
- Rebalancing frequency and costs matter.
- Momentum can reverse sharply.
- Trend signals should be tested across regimes.

Questions to answer after running the notebook:

1. Did the net strategy beat the benchmark?
2. How large was turnover?
3. Which stocks have the strongest current momentum?
4. What risk control would you add?